# Diabetic Retinopathy Grading
Train EfficientNet-B3 on the preprocessed dataset with stratified splits and weighted sampling.


In [1]:
import os

import numpy as np
import pandas as pd
from pathlib import Path
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T

import timm
from sklearn.model_selection import train_test_split


/user/HS400/zv00033/miniconda3/envs/diab/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


## 1. Load Labels & Filter to Existing Images

In [3]:
TRAIN_DIR = Path("dataset/train")
LABELS_CSV = Path("trainLabels.csv")
NUM_CLASSES = 5
IMG_SIZE = 300  

df = pd.read_csv(LABELS_CSV)
# Build full file paths and keep only rows whose images actually exist
df["filepath"] = df["image"].apply(lambda x: str(TRAIN_DIR / f"{x}.jpeg"))
df = df[df["filepath"].apply(os.path.exists)].reset_index(drop=True)

print(f"Total images found: {len(df)}")
print(f"\nClass distribution:\n{df['level'].value_counts().sort_index()}")

Total images found: 35126

Class distribution:
level
0    25810
1     2443
2     5292
3      873
4      708
Name: count, dtype: int64


## 2. Stratified Train / Val / Test Split (70 / 10 / 20)

In [4]:
# First split: 70% train, 30% temp (which becomes 20% test + 10% val)
train_df, temp_df = train_test_split(
    df, test_size=0.30, stratify=df["level"], random_state=42
)

# Second split: split the 30% into 20% test and 10% val  (2/3 and 1/3 of 30%)
test_df, val_df = train_test_split(
    temp_df, test_size=1/3, stratify=temp_df["level"], random_state=42
)

train_df = train_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)

print(f"Train: {len(train_df)}  Test: {len(test_df)}  Val: {len(val_df)}")
print(f"\nTrain class distribution:\n{train_df['level'].value_counts().sort_index()}")
print(f"\nTest class distribution:\n{test_df['level'].value_counts().sort_index()}")
print(f"\nVal class distribution:\n{val_df['level'].value_counts().sort_index()}")

Train: 24588  Test: 7025  Val: 3513

Train class distribution:
level
0    18067
1     1710
2     3704
3      611
4      496
Name: count, dtype: int64

Test class distribution:
level
0    5162
1     489
2    1058
3     175
4     141
Name: count, dtype: int64

Val class distribution:
level
0    2581
1     244
2     530
3      87
4      71
Name: count, dtype: int64


## 3. Class Weights for Loss-Based Balancing
Using cross entropy loss due to the class imbalance


In [5]:
class_counts = train_df["level"].value_counts().sort_index().values
class_weights = 1.0 / class_counts
class_weights = class_weights / class_weights.sum()  # normalize to sum to 1
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float32).to(device)

criterion = nn.CrossEntropyLoss(weight=class_weights_tensor)

print("Using loss-based balancing (weighted CrossEntropyLoss).")
print(f"Class counts: {dict(enumerate(class_counts))}")
print(f"Normalized class weights: {dict(enumerate(class_weights.round(6)))}")


Using loss-based balancing (weighted CrossEntropyLoss).
Class counts: {0: np.int64(18067), 1: np.int64(1710), 2: np.int64(3704), 3: np.int64(611), 4: np.int64(496)}
Normalized class weights: {0: np.float64(0.01213), 1: np.float64(0.128163), 2: np.float64(0.059168), 3: np.float64(0.358688), 4: np.float64(0.441851)}


## 4. Custom Dataset & Data Augmentations

In [6]:
class DRDatasetFromPaths(Dataset):
    """Dataset that loads images from disk."""

    def __init__(self, filepaths, labels, transform=None):
        self.filepaths = filepaths
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        image = Image.open(self.filepaths[idx]).convert("RGB")
        label = int(self.labels[idx])
        if self.transform:
            image = self.transform(image)
        return image, label


### Tranforming the datasets

In [14]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]       #ImageNet normalization for pretrained EfficientNet
IMAGENET_STD  = [0.229, 0.224, 0.225]       #ImageNet normalization for pretrained EfficientNet

train_transform = T.Compose([
    T.RandomRotation(degrees=360),          #Allowing images to be rotated 360 degrees
    T.RandomHorizontalFlip(p=0.5),
    T.RandomVerticalFlip(p=0.5),
    T.ToTensor(),
    T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

eval_transform = T.Compose([
    T.ToTensor(),
    T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

# --- Build datasets -------------------------------------------------------
train_dataset = DRDatasetFromPaths(train_df["filepath"].tolist(), train_df["level"].tolist(), transform=train_transform)
test_dataset  = DRDatasetFromPaths(test_df["filepath"].tolist(), test_df["level"].tolist(), transform=eval_transform)
val_dataset   = DRDatasetFromPaths(val_df["filepath"].tolist(), val_df["level"].tolist(), transform=eval_transform)

print(f"Train dataset size: {len(train_dataset)}")
print(f"Test  dataset size: {len(test_dataset)}")
print(f"Val   dataset size: {len(val_dataset)}")


Train dataset size: 24588
Test  dataset size: 7025
Val   dataset size: 3513


## 5. DataLoaders

Load the datasets into each variable - train_loader, val_loader, test_loader for training and testing.

In [8]:
BATCH_SIZE = 32
NUM_WORKERS = 4

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

## 6. Load EfficientNet-B5 from timm

In [15]:
model = timm.create_model("efficientnet_b5", pretrained=True, num_classes=NUM_CLASSES)
model = model.to(device)

print(model.default_cfg["input_size"])  #Expected input size of model
print(f"Classifier: {model.classifier}")    #Type of classifer model
print(f"Total parameters: {sum(p.numel() for p in model.parameters()):,}")  #Total parameters of model

(3, 448, 448)
Classifier: Linear(in_features=2048, out_features=5, bias=True)
Total parameters: 28,351,029
